# Multilingual Text-JEPA Representation Geometry — Colab Runner
Thin 4-cell wrapper around `src/` and `scripts/`. Cached layer tensors stay on fast local Colab SSD (`data/cache/`), while final tables and 300-DPI figures are saved directly to Google Drive.

In [1]:
# Cell 1: Mount Google Drive, clone/sync repository, and install dependencies
import getpass
import os
from google.colab import drive

drive.mount('/content/drive')

GITHUB_USER = 'alisakour'
REPO_NAME = 'multilingual-text-jepa-geometry'
REPO_DIR = f'/content/{REPO_NAME}'
DRIVE_OUT = '/content/drive/MyDrive/jepa_platonic_benchmark_v2'
os.makedirs(DRIVE_OUT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    token = getpass.getpass('GitHub PAT (required if private repo; leave blank if public): ').strip()
    if token:
        repo_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        repo_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    !git clone {repo_url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

Mounted at /content/drive
GitHub PAT (required if private repo; leave blank if public): ··········
Cloning into '/content/multilingual-text-jepa-geometry'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 73 (delta 24), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 38.12 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/multilingual-text-jepa-geometry


In [2]:
# Cell 2: Build the strictly quad-parallel mLAMA benchmark (AR, EN, ZH, FR)
!python -m src.data.build_benchmark --raw_dir data/raw --out_dir data/processed

Extracting data/raw/mlama1.1.zip -> data/raw ...
Saved quad-parallel benchmark -> data/processed/mlama_quad_parallel_benchmark.csv
Quad-Parallel N=1725 (R=25) | Global=1170, Local=555 | Matched 1:1 Split=555 per domain


In [3]:
# Cell 3: Extract & cache all 13 layers ([CLS] + MeanPool) on GPU (~3-4 min total)
!python scripts/run_extraction.py --config configs/models.yaml --cache_dir data/cache

Extraction device: cuda | Unique checkpoints: 8 | N=1725
Extracting representations: FacebookAI/xlm-roberta-base (poolings=['cls', 'mean'])
config.json: 100% 615/615 [00:00<00:00, 2.77MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 85.5kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 66.5MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 117MB/s]

model.safetensors: downloading bytes:  60% 669M/1.12G [00:02<00:01, 304MB/s, 55.8MB/s  ]
model.safetensors: downloading bytes:  67% 745M/1.12G [00:08<00:11, 32.9MB/s, 40.9MB/s  ]
model.safetensors: reconstructing file:  90% 1.01G/1.12G [00:13<00:01, 85.3MB/s, 21.7MB/s  ]
model.safetensors: downloading bytes: 100% 745M/745M [00:13<00:00, 55.0MB/s, 42.4MB/s  ]
model.safetensors: reconstructing file: 100% 1.12G/1.12G [00:13<00:00, 82.4MB/s, 79.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4199.33it/s]
Saved cache -> data/cache/FacebookAI__xlm-roberta-base.npz
Extracting representations: maktzgls/bert-jepa-infonce-

In [4]:
# Cell 4: Compute Tables 1-5 (BH-FDR) and 300-DPI Figures 1-2 (PNG + PDF) to Google Drive
!python scripts/run_analysis.py --cache_dir data/cache --out_dir "{DRIVE_OUT}"



Running layer-wise geometric evaluation across 9 model configurations...
  [Done] XLM-R [CLS] (L1..L12)
  [Done] XLM-R [MeanPool] (L1..L12)
  [Done] JEPA Mono-15L (FLORES) (L1..L12)
  [Done] JEPA Biling-15L (FLORES) (L1..L12)
  [Done] JEPA Mono-15L (OPUS) (L1..L12)
  [Done] JEPA Biling-15L (OPUS) (L1..L12)
  [Done] JEPA 5L-InfoNCE (OPUS) (L1..L12)
  [Done] JEPA 5L-SIGReg (OPUS) (L1..L12)
  [Done] mE5-Base (Upper-Bound) (L1..L12)

TABLE 1: TERMINAL-LAYER PLATONIC BENCHMARK (VS. XLM-R [CLS], BH-FDR)
          Language Pair                                         Model Mutual 10-NN (%) [FDR] Unrotated Cos (I) Procrustes Cos (R*) [FDR] Rotation Gain (ΔCos) 5-Fold Top-1 / Top-5 (R*) Eff. Spectral Rank
    AR <-> EN (Semitic)            1a. XLM-R Base (Token MLM — [CLS])    17.88 ± 0.38 (Base)     0.176 ± 0.011      0.440 ± 0.005 (Base)               +0.265             24.1% / 43.1%              261.4
    AR <-> EN (Semitic) 1b. XLM-R Base (Token MLM — MeanPool Control)        28.06 ± 0.46**

In [5]:
!python scripts/run_analysis.py --cache_dir data/cache --pooling_override mean --out_dir "{DRIVE_OUT}/meanpool_control" --no_plots

Running layer-wise geometric evaluation across 9 model configurations...
  [Done] XLM-R [CLS] (L1..L12)
  [Done] XLM-R [MeanPool] (L1..L12)
  [Done] JEPA Mono-15L (FLORES) (L1..L12)
  [Done] JEPA Biling-15L (FLORES) (L1..L12)
  [Done] JEPA Mono-15L (OPUS) (L1..L12)
  [Done] JEPA Biling-15L (OPUS) (L1..L12)
  [Done] JEPA 5L-InfoNCE (OPUS) (L1..L12)
  [Done] JEPA 5L-SIGReg (OPUS) (L1..L12)
  [Done] mE5-Base (Upper-Bound) (L1..L12)

TABLE 1: TERMINAL-LAYER PLATONIC BENCHMARK (VS. XLM-R [CLS], BH-FDR)
          Language Pair                                         Model Mutual 10-NN (%) [FDR] Unrotated Cos (I) Procrustes Cos (R*) [FDR] Rotation Gain (ΔCos) 5-Fold Top-1 / Top-5 (R*) Eff. Spectral Rank
    AR <-> EN (Semitic)            1a. XLM-R Base (Token MLM — [CLS])    28.06 ± 0.46 (Base)     0.365 ± 0.005      0.629 ± 0.005 (Base)               +0.263             49.8% / 74.8%              333.6
    AR <-> EN (Semitic) 1b. XLM-R Base (Token MLM — MeanPool Control)        28.06 ± 0.46 n